# 📘 Notebook 04 · Web3 入门 + 区块链原理

> 学 Web3 的最大坑是从"DeFi 协议怎么用"开始学。这跟你**没碰过编程就想用 PyTorch** 一样。我们从底层开始，**手写一个区块链**。

**学完你会：**

- 真正理解区块链 = 哈希链 + 数字签名 + 共识 三件事
- 用 Python 手写一个能挖矿、能签名转账的最简区块链
- 理解 Merkle 树、UTXO vs Account Model
- 理解为什么以太坊比比特币更适合做应用
- 知道 Gas、EVM、Nonce 这些术语真实指什么

**预计时间：6-8 小时**

**前置要求**：Python 基础。不需要密码学背景，我们边写边讲。

## 1. Web3 是什么（去除一切炒作之后）

### 1.1 一句话

> **Web3 = 用区块链作为后端的应用 + 用代币做激励的协调机制。**

去掉"价值互联网""未来""革命"这些词，剩下的就这两件事。

### 1.2 三个时代

| 时代 | 特征 | 经济模式 |
|---|---|---|
| **Web1** (1990s-2005) | 静态网页、单向阅读 | 广告（Yahoo） |
| **Web2** (2005-2020) | 用户产生内容、平台聚合 | 平台抽成（FB、Google）|
| **Web3** (2020-?) | 用户掌握数据 / 资产 | 协议费用 + 代币 |

Web2 的核心矛盾：**平台拥有用户数据和价值，用户没有议价权**。Web3 试图解决这个，方法是把"数据库"换成"区块链"。

### 1.3 是什么不是什么

✅ Web3 **是**：
- 一种应用架构（后端跑在区块链上）
- 一种金融基础设施（无须信任的清结算）
- 一种用户身份方案（钱包代替账号密码）
- 一种社区协调机制（DAO、代币激励）

❌ Web3 **不是**：
- "互联网的未来"——更可能是"互联网的一种新形态"
- "去中介化一切"——很多场景中心化反而更高效
- "一定能颠覆 Web2"——目前 90% 应用还在试错
- "区块链 = Web3"——区块链是底层技术之一

### 1.4 谁在这个行业

```
基础设施
  ├─ 公链：以太坊、Solana、Sui、Aptos、Bitcoin、各种 L2
  ├─ 节点服务：Alchemy、Infura、QuickNode
  ├─ 索引：The Graph、Dune
  └─ 钱包：MetaMask、Phantom、Rabby

DeFi（去中心化金融，体量最大）
  ├─ DEX：Uniswap、Curve、PancakeSwap
  ├─ 借贷：Aave、Compound、MakerDAO
  ├─ 衍生品：dYdX、GMX、Hyperliquid
  └─ Yield：Yearn、Convex

NFT / 游戏
  ├─ NFT 市场：OpenSea、Blur、Magic Eden
  └─ 链游：Axie、Pixels

其他
  ├─ 中心化交易所（CEX）：Binance、Coinbase、OKX、Bybit
  ├─ 做市商：Wintermute、GSR、Cumberland
  └─ 链上量化 / MEV 团队：Flashbots、各种独立团队
```

## 2. 区块链 = 三件事

把"区块链"这个词解构成三个**可独立实现**的部分，会清楚很多：

### 2.1 哈希链：让数据不可篡改

把数据组织成"区块"，每个区块包含**上一个区块的哈希值**。改一个字节，后面所有哈希全错。

```
区块 0          区块 1               区块 2
[data, hash0]→[data, hash0, hash1]→[data, hash1, hash2]→...
```

**Python 立即可验证：**

In [ ]:
import hashlib
import json

def hash_block(block: dict) -> str:
    """对一个 block 算 SHA256"""
    encoded = json.dumps(block, sort_keys=True).encode()
    return hashlib.sha256(encoded).hexdigest()

# 创建链
genesis = {
    'index': 0,
    'data': '创世块',
    'prev_hash': '0' * 64,
}
genesis_hash = hash_block(genesis)
print('创世块 hash:', genesis_hash[:16], '...')

# 第二个块
block1 = {
    'index': 1,
    'data': 'Alice 转 Bob 10 元',
    'prev_hash': genesis_hash,
}
block1_hash = hash_block(block1)
print('块 1 hash:', block1_hash[:16], '...')

# 篡改创世块
genesis_tampered = dict(genesis)
genesis_tampered['data'] = '我是黑客'
print('\n篡改后创世块 hash:', hash_block(genesis_tampered)[:16], '...')
print('与原来相比：', '不同' if hash_block(genesis_tampered) != genesis_hash else '相同')
print('于是块 1 的 prev_hash 对不上了，链断了')

**关键理解**：

- "不可篡改" 不是"物理上不能改"，而是"改了会被立刻发现"
- 单机环境下，黑客可以把整条链都重算一遍。所以需要第三件事——**分布式共识**。
- 但如果只是一个人记账，哈希链已经足够。这就是 git 的设计——**git 本质上就是个版本化的哈希链**。

### 2.2 数字签名：让转账无法伪造

如果只有哈希链，"Alice 给 Bob 转 10 元" 这种交易**任何人都能写**。我们需要让交易只能被 Alice 本人发出。

**非对称加密**：每个人有一对密钥
- **私钥**：自己保管，绝不能泄露
- **公钥**：可以公开，相当于身份证

签名：用**私钥**对消息加密 → 签名
验证：用**公钥**验证签名是否对得上消息

下面用 `ecdsa` 库做实际签名（椭圆曲线签名算法，比特币和以太坊都用 secp256k1 曲线）。

In [ ]:
# 安装：pip install ecdsa
# (如果你没装，下面代码会报 ImportError；先跳过看后面也行)
try:
    from ecdsa import SigningKey, VerifyingKey, SECP256k1, BadSignatureError
    HAS_ECDSA = True
except ImportError:
    HAS_ECDSA = False
    print('未安装 ecdsa：pip install ecdsa')

if HAS_ECDSA:
    # ---------- Alice 生成密钥对 ----------
    alice_sk = SigningKey.generate(curve=SECP256k1)
    alice_pk = alice_sk.get_verifying_key()

    print('Alice 私钥（hex 前 32 位）:', alice_sk.to_string().hex()[:32], '...')
    print('Alice 公钥（hex 前 32 位）:', alice_pk.to_string().hex()[:32], '...')

    # ---------- Alice 签一笔交易 ----------
    tx = b'Alice -> Bob : 10 coins'
    signature = alice_sk.sign(tx)
    print('\n签名:', signature.hex()[:32], '...')

    # ---------- 任何人都能验证 ----------
    try:
        alice_pk.verify(signature, tx)
        print('验证成功：这笔交易确实是 Alice 签的')
    except BadSignatureError:
        print('验证失败')

    # ---------- 篡改交易 ----------
    tampered_tx = b'Alice -> Eve : 1000 coins'
    try:
        alice_pk.verify(signature, tampered_tx)
        print('验证成功')
    except BadSignatureError:
        print('\n篡改后验证失败：签名对不上消息')

**这一段是 Web3 最重要的概念之一：**

- 你的**钱包**就是一对密钥
- **地址** = 公钥的哈希（取前 20 字节，对以太坊）
- **钱包密码不是真正的密码**，它只是用来加密本地保存的私钥；私钥才是身份本身
- 私钥丢了 = 钱包永久丢失，没有"找回密码"

**助记词**（12 / 24 个英文单词）是私钥的"种子"，可以通过 BIP-39 算法转回私钥。

### 2.3 共识：让多台机器同步同一份链

哈希链 + 数字签名足够让单机记账安全。但分布式系统里，每台机器都有自己的账本，要怎么同步？

**共识算法**做的就是这件事：让网络中大部分机器**对账本达成一致**。

**两个主流共识：**

| 共识 | 怎么做 | 代表 |
|---|---|---|
| **PoW**（工作量证明） | 比谁先算出符合条件的哈希（挖矿）| 比特币、以太坊（合并前）|
| **PoS**（权益证明） | 按持币比例选出块生产者 | 以太坊（合并后）、Solana、Cardano |

我们演示 PoW，因为更直观。

In [ ]:
def pow_mine(block: dict, difficulty: int = 4):
    """
    工作量证明：找到一个 nonce，让 hash(block) 以 difficulty 个 0 开头
    difficulty 越大越难（每加一个 0，难度大约乘以 16）
    """
    target = '0' * difficulty
    nonce = 0
    while True:
        block['nonce'] = nonce
        h = hash_block(block)
        if h.startswith(target):
            return nonce, h
        nonce += 1

import time

block = {
    'index': 1,
    'data': 'Alice -> Bob : 10 coins',
    'prev_hash': '0' * 64,
}

for difficulty in [2, 3, 4, 5]:
    block_copy = dict(block)
    t0 = time.time()
    nonce, h = pow_mine(block_copy, difficulty=difficulty)
    dt = time.time() - t0
    print(f'难度 {difficulty} ({difficulty}个0): nonce={nonce:>8}  hash={h[:16]}...  耗时 {dt:.3f}s')

**关键理解**：

- 难度从 4 到 5，时间大约 ×16。比特币的难度是要 hash 以约 19 个 0 开头，工程量 = 全世界矿机加起来 10 分钟才能算出
- **51% 攻击**：如果有人控制了全网 51% 算力，他可以重写历史。所以全球分布、算力分散是关键
- **PoW 的问题**：极度耗能。比特币年耗电 ≈ 阿根廷全国
- **PoS 的解决**：不挖矿，按持币比例选出块生产者；作恶会被罚没质押的币

## 3. 把三件事拼起来：手写一个区块链

下面是一个**功能完整**的简化区块链：
- 创建钱包、签名转账
- 把交易打包进块、挖矿
- 维护账本余额、防双花
- 验证整条链的有效性

代码不到 100 行。

In [ ]:
import hashlib
import json
import time
from collections import defaultdict

if HAS_ECDSA:
    class Wallet:
        """钱包 = 一对密钥 + 地址"""
        def __init__(self):
            self.sk = SigningKey.generate(curve=SECP256k1)
            self.pk = self.sk.get_verifying_key()
            # 地址 = 公钥的 SHA256，取前 20 字节
            self.address = hashlib.sha256(self.pk.to_string()).hexdigest()[:40]

        def sign(self, message: bytes) -> bytes:
            return self.sk.sign(message)

        @staticmethod
        def verify(pk_bytes: bytes, message: bytes, signature: bytes) -> bool:
            pk = VerifyingKey.from_string(pk_bytes, curve=SECP256k1)
            try:
                pk.verify(signature, message)
                return True
            except BadSignatureError:
                return False


    class Transaction:
        """一笔转账交易"""
        def __init__(self, sender_pk: bytes, sender_addr: str,
                     receiver_addr: str, amount: float, nonce: int):
            self.sender_pk    = sender_pk.hex()
            self.sender_addr  = sender_addr
            self.receiver_addr = receiver_addr
            self.amount       = amount
            self.nonce        = nonce      # 防重放
            self.signature    = None

        def to_dict(self, include_sig=True):
            d = {
                'sender_pk': self.sender_pk,
                'sender_addr': self.sender_addr,
                'receiver_addr': self.receiver_addr,
                'amount': self.amount,
                'nonce': self.nonce,
            }
            if include_sig and self.signature:
                d['signature'] = self.signature
            return d

        def hash(self) -> bytes:
            """待签名的内容"""
            return json.dumps(self.to_dict(include_sig=False), sort_keys=True).encode()

        def sign_with(self, wallet: 'Wallet'):
            assert wallet.address == self.sender_addr
            self.signature = wallet.sign(self.hash()).hex()

        def is_valid(self) -> bool:
            if not self.signature:
                return False
            try:
                pk_bytes = bytes.fromhex(self.sender_pk)
                return Wallet.verify(pk_bytes, self.hash(), bytes.fromhex(self.signature))
            except Exception:
                return False


    class Block:
        def __init__(self, index: int, transactions: list, prev_hash: str):
            self.index        = index
            self.transactions = transactions
            self.prev_hash    = prev_hash
            self.timestamp    = time.time()
            self.nonce        = 0

        def to_dict(self):
            return {
                'index': self.index,
                'transactions': [tx.to_dict() for tx in self.transactions],
                'prev_hash': self.prev_hash,
                'timestamp': self.timestamp,
                'nonce': self.nonce,
            }

        def hash(self) -> str:
            return hashlib.sha256(json.dumps(self.to_dict(), sort_keys=True).encode()).hexdigest()

        def mine(self, difficulty: int):
            target = '0' * difficulty
            while not self.hash().startswith(target):
                self.nonce += 1


    class Blockchain:
        def __init__(self, difficulty: int = 3):
            self.chain      = [self._genesis()]
            self.pending    = []
            self.difficulty = difficulty
            self.balances   = defaultdict(float)
            self.nonces     = defaultdict(int)    # 防重放：每个地址下一个允许的 nonce
            # 初始资金：给一个"系统"地址凭空铸造（模拟创世）
            self.balances['SYSTEM'] = 1_000_000

        def _genesis(self):
            b = Block(0, [], '0' * 64)
            return b

        def add_tx(self, tx: Transaction):
            # 验证签名
            if not tx.is_valid():
                raise ValueError('签名无效')
            # 验证余额
            if self.balances[tx.sender_addr] < tx.amount:
                raise ValueError(f'余额不足: {tx.sender_addr} 有 {self.balances[tx.sender_addr]}, 转 {tx.amount}')
            # 验证 nonce（防重放攻击）
            if tx.nonce != self.nonces[tx.sender_addr]:
                raise ValueError(f'nonce 错误: 期望 {self.nonces[tx.sender_addr]}, 给的 {tx.nonce}')
            self.pending.append(tx)

        def mine_pending(self, miner_addr: str, reward: float = 50):
            # 给矿工挖矿奖励（凭空铸造，等于通胀）
            coinbase = Transaction(b'', 'SYSTEM', miner_addr, reward, nonce=-1)
            coinbase.signature = 'coinbase'    # 特殊：coinbase 不要签名

            block = Block(
                index=len(self.chain),
                transactions=[coinbase] + self.pending,
                prev_hash=self.chain[-1].hash(),
            )
            block.mine(self.difficulty)
            self.chain.append(block)

            # 更新余额
            for tx in self.pending:
                self.balances[tx.sender_addr] -= tx.amount
                self.balances[tx.receiver_addr] += tx.amount
                self.nonces[tx.sender_addr] += 1
            self.balances[miner_addr] += reward
            self.pending = []
            return block

        def validate(self) -> bool:
            """验证整条链"""
            for i in range(1, len(self.chain)):
                b = self.chain[i]
                # 1. prev_hash 必须对得上
                if b.prev_hash != self.chain[i-1].hash():
                    print(f'块 {i} prev_hash 错误'); return False
                # 2. 工作量证明
                if not b.hash().startswith('0' * self.difficulty):
                    print(f'块 {i} 工作量证明无效'); return False
                # 3. 块内每笔交易（非 coinbase）签名有效
                for tx in b.transactions:
                    if tx.signature == 'coinbase':
                        continue
                    if not tx.is_valid():
                        print(f'块 {i} 中存在无效交易'); return False
            return True
    print('类定义完毕')
else:
    print('需要 ecdsa 才能跑下面的演示')

In [ ]:
if HAS_ECDSA:
    # ---------- 跑一遍 ----------
    chain = Blockchain(difficulty=3)

    # 创建三个钱包
    alice = Wallet()
    bob   = Wallet()
    miner = Wallet()

    print('Alice 地址:', alice.address[:16], '...')
    print('Bob 地址  :', bob.address[:16], '...')
    print('矿工地址  :', miner.address[:16], '...')

    # 系统先给 Alice 1000 元（模拟空投）
    chain.balances[alice.address] = 1000

    # Alice 转 30 给 Bob
    tx1 = Transaction(alice.pk.to_string(), alice.address, bob.address, 30, nonce=0)
    tx1.sign_with(alice)
    chain.add_tx(tx1)
    print('\n[第 1 笔] Alice → Bob 30')

    # Alice 又转 50 给 Bob
    tx2 = Transaction(alice.pk.to_string(), alice.address, bob.address, 50, nonce=1)
    tx2.sign_with(alice)
    chain.add_tx(tx2)
    print('[第 2 笔] Alice → Bob 50')

    # 矿工打包并挖出块
    block = chain.mine_pending(miner.address, reward=50)
    print(f'\n矿工挖出块 #{block.index}  nonce={block.nonce}  hash={block.hash()[:16]}...')

    # 看看余额
    print(f'\nAlice 余额: {chain.balances[alice.address]}')
    print(f'Bob   余额: {chain.balances[bob.address]}')
    print(f'矿工  余额: {chain.balances[miner.address]}')

    # 验证整条链
    print(f'\n整条链是否有效: {chain.validate()}')

In [ ]:
if HAS_ECDSA:
    # ---------- 攻击演示 1：重放攻击 ----------
    # 黑客 Eve 截获 Alice 的 tx1，想再发一次
    print('=== 重放攻击测试 ===')
    try:
        chain.add_tx(tx1)    # nonce=0 已经用过了
    except ValueError as e:
        print('防住了:', e)

    # ---------- 攻击演示 2：伪造交易 ----------
    print('\n=== 伪造攻击测试 ===')
    fake = Transaction(alice.pk.to_string(), alice.address, 'eve_addr', 9999, nonce=2)
    # Eve 想假装 Alice 签名，但她没有 Alice 的私钥，签名出来不对
    fake.signature = 'aabbccdd' * 16     # 瞎写
    try:
        chain.add_tx(fake)
    except ValueError as e:
        print('防住了:', e)

    # ---------- 攻击演示 3：篡改已有块 ----------
    print('\n=== 篡改链测试 ===')
    chain.chain[1].transactions[1].amount = 10000   # 篡改第二个块第二笔交易
    print('整条链是否有效:', chain.validate())

**🎉 恭喜你刚刚手写了一个完整的区块链。**

它包含了真实区块链的所有核心要素：
- ✅ 哈希链
- ✅ 数字签名（ECDSA / secp256k1，和比特币以太坊用的是同一种）
- ✅ 工作量证明
- ✅ 防重放（nonce）
- ✅ 防双花（余额校验）
- ✅ 挖矿奖励（coinbase）
- ✅ 全链验证

只比真实比特币少了：
- 分布式网络（P2P + Gossip 协议）
- UTXO 模型（我们用了更简单的 account model）
- 难度自动调整
- 经济激励的博弈分析

但**理解了上面 100 行代码，你已经理解了 90% 的区块链原理**。

### 🎯 练习

三个练习：

1. 加一个"链分叉"功能：如果有两个矿工同时挖出块，按"最长链原则"选择哪条
2. 实现一个简化的 P2P：用两个 Blockchain 实例模拟两个节点，写一个 `sync(peer)` 方法让它们同步
3. 加上"难度自动调整"：根据最近 10 个块的平均挖矿时间，调整 difficulty 让平均出块时间稳定在 5 秒

In [ ]:
# 在这里写你的答案


## 4. UTXO vs Account：两种账户模型

我们刚才用的是 **Account Model**（账户模型），以太坊就用这个。

**比特币用的是 UTXO Model**（Unspent Transaction Output），完全不同。

### 4.1 Account Model（以太坊、Solana）

像银行账户：
- 每个地址有一个余额
- 转账 = 把发送方余额 -= amount，接收方余额 += amount
- 状态简单，写智能合约方便

### 4.2 UTXO Model（比特币）

像现金：
- 钱不存在"余额"里，存在"未花费的输出"（UTXO）里
- 转账 = "花掉"几个 UTXO，"创建"新的 UTXO 给接收方
- 每笔交易由 inputs（消耗的 UTXO）+ outputs（新建的 UTXO）组成

### 4.3 一个例子

> Alice 有两个 UTXO，分别值 5 元、3 元。要转 6 元给 Bob。

**UTXO 模型下：**
- Inputs: 两个 UTXO（5 + 3 = 8）
- Outputs:
  - 6 元给 Bob
  - 2 元找零给自己（Alice）

像现金消费：你付一张 5 元一张 3 元买 6 元东西，找回 2 元。

**Account 模型下：**
- Alice 余额 -= 6
- Bob 余额 += 6

简单得多。

### 4.4 两种模型的权衡

| 维度 | UTXO | Account |
|---|---|---|
| 并发处理 | 容易（不同 UTXO 互不冲突）| 难（同一账户冲突）|
| 隐私 | 好（每笔交易可用新地址）| 差（地址固定）|
| 智能合约 | 难写 | 容易 |
| 状态存储 | 大（要存所有 UTXO）| 小 |
| 用户心智模型 | 反直觉 | 直觉 |

以太坊为了支持复杂的智能合约，选择了 Account Model。这也是它能孵化出整个 DeFi 生态的根本原因之一。

## 5. Merkle 树：高效验证

如果一个块里有 1 万笔交易，验证"某笔交易在块里"需要扫描整个块。**Merkle 树**让验证变成 O(log n)。

### 5.1 构造

- 叶子：每笔交易的哈希
- 内部节点：两个子节点哈希的哈希
- 根：整棵树的根哈希

```
              [Root]
             /      \
        [h12]        [h34]
        /   \        /   \
      [h1] [h2]    [h3] [h4]
       |    |       |    |
      tx1  tx2     tx3  tx4
```

### 5.2 Merkle Proof（默克尔证明）

要证明 tx2 在树里，只需要给：
- tx2 本身
- h1（用于算出 h12）
- h34（用于和 h12 一起算出 root）

验证者算：`hash(hash(h1, hash(tx2)), h34)` == root？

只需 **log₂(n)** 个节点。1 万笔交易只需要 ~14 个哈希。

### 5.3 应用

- **轻节点**：手机钱包不存全链，只存块头（含 Merkle Root），靠 Merkle Proof 验证某笔交易
- **跨链桥**：A 链证明"在 B 链上 Alice 收到了 100 元"用 Merkle Proof
- **零知识证明**：Merkle 树是很多 zk 系统的基础结构

In [ ]:
def hash_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def hash_pair(a: str, b: str) -> str:
    return hash_bytes((a + b).encode())

class MerkleTree:
    def __init__(self, leaves: list):
        if not leaves:
            self.root = hash_bytes(b'')
            return
        self.leaves = [hash_bytes(x.encode()) if isinstance(x, str) else hash_bytes(x) for x in leaves]
        self.layers = [self.leaves[:]]
        # 自底向上构造
        cur = self.leaves[:]
        while len(cur) > 1:
            if len(cur) % 2 == 1:
                cur.append(cur[-1])    # 奇数补最后一个
            cur = [hash_pair(cur[i], cur[i+1]) for i in range(0, len(cur), 2)]
            self.layers.append(cur)
        self.root = cur[0]

    def proof(self, index: int) -> list:
        """返回某个叶子的 Merkle Proof：(sibling_hash, is_left) 的列表"""
        proof = []
        idx = index
        for layer in self.layers[:-1]:
            if idx % 2 == 0:
                sibling = layer[idx + 1] if idx + 1 < len(layer) else layer[idx]
                proof.append((sibling, 'right'))
            else:
                sibling = layer[idx - 1]
                proof.append((sibling, 'left'))
            idx //= 2
        return proof

    @staticmethod
    def verify(leaf_data, proof: list, root: str) -> bool:
        h = hash_bytes(leaf_data.encode()) if isinstance(leaf_data, str) else hash_bytes(leaf_data)
        for sibling, side in proof:
            if side == 'right':
                h = hash_pair(h, sibling)
            else:
                h = hash_pair(sibling, h)
        return h == root

# 构造一棵树
txs = [f'tx_{i}' for i in range(8)]
tree = MerkleTree(txs)
print(f'8 笔交易，Merkle Root = {tree.root[:16]}...')

# 证明 tx_3 在树里
proof = tree.proof(3)
print(f'\ntx_3 的 Merkle Proof（{len(proof)} 个哈希）:')
for h, side in proof:
    print(f'  {h[:16]}...  ({side})')

# 验证
ok = MerkleTree.verify('tx_3', proof, tree.root)
print(f'\n验证 tx_3 在树中: {ok}')

# 伪造一个不在树里的交易
fake_ok = MerkleTree.verify('tx_evil', proof, tree.root)
print(f'伪造交易能通过验证吗: {fake_ok}')

## 6. 从比特币到以太坊：为什么 Web3 是以太坊带起来的

### 6.1 比特币的限制

比特币的脚本系统**故意设计得不图灵完备**——只能做有限的几种操作（验证签名、时间锁等）。

好处：安全、无限循环不会让网络瘫痪。
坏处：写不了复杂应用。

### 6.2 以太坊的革命：智能合约 + EVM

Vitalik 2013 年的想法：把比特币的脚本换成一个**图灵完备的虚拟机**（EVM）。任何人都能在链上部署一段代码（智能合约），这段代码自动执行、不可篡改。

```
传统应用：              以太坊应用 (DApp)：
[前端] ↔ [后端 API] ↔ [数据库]
                       ↑
                       公司控制
                       可以删数据
                       可以改逻辑

[前端] ↔ [区块链] ← 智能合约存在这里
                       ↑
                       任何人验证
                       不能删 / 改
                       自动执行
```

### 6.3 关键概念

| 概念 | 含义 | 类比 |
|---|---|---|
| **EVM** | 以太坊虚拟机 | 类似 JVM、Python 解释器 |
| **Gas** | 执行操作的"燃料费" | 算力收费 |
| **Gas Price** | 你愿意为每单位 Gas 付多少 ETH | 给小费让矿工优先打包 |
| **Nonce** | 账户的交易序号，从 0 开始 | 防重放 + 排序 |
| **合约地址** | 智能合约的"地址" | 程序的入口 |
| **ABI** | 应用二进制接口，描述合约函数 | 类似 OpenAPI |

### 6.4 一笔以太坊交易长什么样

```python
{
    'nonce': 42,                                    # 我的第 42 笔交易
    'gasPrice': 30_000_000_000,                     # 30 gwei
    'gasLimit': 21_000,                             # 简单转账用 21000
    'to': '0x71C7656EC7ab88b098defB751B7401B5f6d8976F',
    'value': 1_000_000_000_000_000_000,              # 1 ETH (10^18 wei)
    'data': b'',                                     # 普通转账为空；调合约时是函数调用编码
    'chainId': 1,                                    # 1 = 主网
    'v', 'r', 's': ...,                              # 签名（ECDSA）
}
```

**这就是 Web3 的"API 调用"长什么样。**

## 7. 接下来去哪儿

到这里，你应该已经：
- ✅ 知道区块链 = 哈希链 + 签名 + 共识
- ✅ 手写过完整区块链
- ✅ 理解 UTXO vs Account 模型
- ✅ 理解 Merkle 树和它的用途
- ✅ 理解以太坊为什么是 Web3 的核心

**下一步：`05_Web3_Solidity_智能合约.ipynb`**

我们会：
- 写并部署第一个 ERC20 代币（你自己的"币"）
- 学 Solidity 的核心语法
- 用 web3.py 与链交互
- 理解 Gas、存储、安全漏洞

**自学检测**：把以下三个问题用自己的话答出来，再进入下一节：

1. 区块链是怎么"不可篡改"的？谁来保证？
2. 你的钱包私钥丢了，为什么没法找回？
3. 以太坊比比特币多做了什么，才能跑 DeFi？